# Links to a setting: agent_api_docs only, or one sentence in the prompt?

`docs/spec/LINKS.md` puts the `[label](window-ml:<address>)` form in `agent_api_docs`'s "Your settings" section and
leaves open (question 1) whether one sentence in the system prompt earns its tokens. The sweep `links-prompt`
(`tests/e2e/bench/specs/links-prompt.bench.ts`) runs the same build in two arms, `__ML_LINKS_SENTENCE__` off and on:

> To point the person at one of the extension's settings, link it as `[label](window-ml:setting/<key>)`;
> `agent_api_docs` lists the keys.

Four tasks ask where something is set (the API key, running programs, auto-approving read-only scripts, backup), which a
link answers; a fifth asks for a code on the page and counts links nobody asked for. Per run:

- **linked**: the answer has at least one `[label](window-ml:…)` link;
- **resolves**: some link's address parses (`parseAddress`, the panel's own parser), so it renders as a button;
- **right place**: some link opens the tab and section the task is about (the spec's pass);
- **row**: some link is the exact `setting/<key>` row (backup has no row: its section counts);
- **bare form**: the answer names `window-ml:` outside a Markdown link, which renders as plain text.

And the sentence's cost: the first call's prompt tokens, on minus off, per model (each model's own tokenizer).

In [1]:
import json, re, subprocess
from pathlib import Path
import pandas as pd
from scipy.stats import fisher_exact
import wmlnb

# The checkout this notebook runs in (the kernel's cwd), not wmlnb.ROOT: a worktree borrowing the clone's venv has its
# ROOT in the clone, and its pinned data would be read from (and written to) there.
TOP = Path(subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True, check=True).stdout.strip())
HERE = TOP / "notebooks" / "bench"
SWEEP = "links-prompt"
EXTRACT = HERE / "data" / "links-prompt" / "runs.json"
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 30)

## The runs, pinned

One row per run, from each run's `cell.json` (outcome, answer) and `run.json` (its steps and each call's usage). Every
address an answer links is resolved at extract time by the product's parser (`src/link-address.ts`), and the result is
pinned with the row, so the notebook does not need the source to run.

In [2]:
WANT = {  # the spec's own `want` per task (links-prompt.bench.ts TASKS)
    "api-key": {"tab": "connection", "key": "apiKey"},
    "shell": {"tab": "advanced", "section": "shell", "key": "shellEnabled"},
    "readonly": {"tab": "advanced", "section": "javascript", "key": "autoApproveReadonly"},
    "backup": {"tab": "appearance", "backup": True},
}
RESOLVE_JS = (
    'import fs from "node:fs"; import { parseAddress } from "./src/link-address.ts"; '
    'const addrs = JSON.parse(fs.readFileSync(0, "utf8")); '
    'console.log(JSON.stringify(Object.fromEntries(addrs.map((a) => [a, parseAddress(a)]))));'
)


def links_in(text):
    return [{"label": m[1], "address": m[2]} for m in re.finditer(r"\[([^\]\n]+)\]\(window-ml:([^)\s]*)\)", text)]


def resolve(addresses):
    out = subprocess.run(["node", "--import", "tsx", "--input-type=module", "-e", RESOLVE_JS],
                         input=json.dumps(sorted(addresses)), capture_output=True, text=True, cwd=TOP, check=True)
    return json.loads(out.stdout)


def read_sweep():
    rows = []
    for cell in sorted(wmlnb.find_sweep(SWEEP).glob("*/*/r*/cell.json")):
        if "history" in cell.parts:
            continue
        c = json.loads(cell.read_text())
        m = c.get("measurement") or {}
        run = cell.parent / "run.json"
        steps = json.loads(run.read_text())["session"]["steps"] if run.exists() else []
        usage = [s["usage"] for s in steps if s.get("usage")]
        answer = m.get("finalAnswer") or m.get("answer") or ""
        rows.append({
            "task": c["taskId"], "sentence": c["combo"]["sentence"], "model": c["combo"]["model"], "repeat": c["repeat"],
            "passed": bool(m.get("succeeded")), "error": bool(m.get("error")), "hit_cap": bool(m.get("hitCap")),
            "steps": m.get("steps"), "calls": len(usage),
            "first_prompt_tokens": usage[0].get("promptTokens") if usage else None,
            "prompt_tokens": (m.get("tokens") or {}).get("prompt"),
            "docs_calls": sum(1 for s in steps if s.get("tool") == "agent_api_docs"),
            "answer": answer, "links": links_in(answer),
        })
    resolved = resolve({l["address"] for r in rows for l in r["links"]})
    for r in rows:
        for l in r["links"]:
            l["route"] = resolved.get(l["address"])
    return rows

rows = wmlnb.extract(EXTRACT, read_sweep, source={"sweep": SWEEP})
runs = pd.DataFrame(rows)
pd.DataFrame(wmlnb.provenance(EXTRACT))

,extract,sha256,rows,extracted_at,commit,sweep
0,.claude/worktrees/links-prompt/notebooks/bench...,1a36ed9c3b1d49cb,200,2026-11-13T16:50:41+00:00,7ded3154a3f9,links-prompt


In [3]:
def opens(route, want):
    if not route:
        return False
    if want.get("backup"):
        return route.get("target") == "set-backup"
    return route.get("tab") == want["tab"] and ("section" not in want or route.get("section") == want["section"])


def judge(r):
    links, want = r["links"], WANT.get(r["task"])
    routes = [l["route"] for l in links]
    return pd.Series({
        "linked": bool(links),
        "resolves": any(routes),
        "right_place": bool(want) and any(opens(x, want) for x in routes),
        "row": bool(want) and any(x and (x.get("setting") == want.get("key") or (want.get("backup") and opens(x, want))) for x in routes),
        "bare_form": "window-ml:" in r["answer"].replace("](window-ml:", ""),
        "unresolved": sum(1 for x in routes if not x),
    })

runs = runs.join(runs.apply(judge, axis=1))
ok = runs[~runs.error]
asks = ok[ok.task != "control"]
print(f"{len(runs)} runs, {int(runs.error.sum())} errored (left out below); {len(asks)} on the four link tasks")

200 runs, 0 errored (left out below); 160 on the four link tasks


## Link rate and correctness, per arm

The four link tasks pooled, per model and arm: share of runs with a link, one that resolves, one to the right place,
and the exact row.

In [4]:
def rates(df, by):
    g = df.groupby(by)
    out = pd.DataFrame({"runs": g.size()})
    for col in ["linked", "resolves", "right_place", "row", "bare_form"]:
        out[col] = g[col].mean().round(2)
    return out

rates(asks, ["model", "sentence"]).unstack("sentence")

runs     linked       resolves       right_place         row       bare_form     
sentence                      off  on    off    on      off    on         off    on   off    on       off   on
model                                                                                                         
deepseek.deepseek-flash        16  16   0.56  0.94     0.56  0.94        0.56  0.94  0.56  0.81       0.0  0.0
gemma4:31b                     16  16   0.56  0.75     0.56  0.75        0.56  0.75  0.25  0.62       0.0  0.0
gemma4:e4b                     16  16   0.06  0.06     0.06  0.06        0.06  0.06  0.06  0.00       0.0  0.0
openrouter.openai/gpt-6-luna   16  16   0.50  0.88     0.50  0.88        0.50  0.88  0.25  0.25       0.0  0.0
qwen3.8-flash-next:vision      16  16   0.88  1.00     0.88  0.94        0.88  0.81  0.69  0.56       0.0  0.0

In [5]:
rates(asks, ["sentence"])

,runs,linked,resolves,right_place,row,bare_form
sentence,,,,,,
off,80,0.51,0.51,0.51,0.36,0.0
on,80,0.72,0.71,0.69,0.45,0.0


Per task, both arms (right place, as a count of runs): a task only one arm gets right points at what the sentence adds.

In [6]:
asks.pivot_table(index=["model", "task"], columns="sentence", values="right_place", aggfunc="sum").astype(int)

sentence                               off  on
model                        task             
deepseek.deepseek-flash      api-key     2   4
                             backup      3   3
                             readonly    4   4
                             shell       0   4
gemma4:31b                   api-key     3   4
                             backup      3   4
                             readonly    3   4
                             shell       0   0
gemma4:e4b                   api-key     0   0
                             backup      0   0
                             readonly    1   1
                             shell       0   0
openrouter.openai/gpt-6-luna api-key     0   4
                             backup      3   4
                             readonly    4   4
                             shell       1   2
qwen3.8-flash-next:vision    api-key     4   4
                             backup      4   4
                             readonly    4   3
                             shell       2   2

## Is the difference more than noise?

Fisher's exact test per model on right-place runs over the four link tasks, two-sided, and pooled over models.

In [7]:
def fisher(df):
    out = []
    for model, g in [*df.groupby("model"), ("(all)", df)]:
        a = g[g.sentence == "off"].right_place; b = g[g.sentence == "on"].right_place
        if not len(a) or not len(b):
            continue
        t = [[a.sum(), len(a) - a.sum()], [b.sum(), len(b) - b.sum()]]
        out.append({"model": model, "off": f"{a.sum()}/{len(a)}", "on": f"{b.sum()}/{len(b)}",
                    "p": round(fisher_exact(t).pvalue, 3)})
    return pd.DataFrame(out).set_index("model")

fisher(asks)

,off,on,p
model,,,
deepseek.deepseek-flash,9/16,15/16,0.037
gemma4:31b,9/16,12/16,0.458
gemma4:e4b,1/16,1/16,1.000
openrouter.openai/gpt-6-luna,8/16,14/16,0.054
qwen3.8-flash-next:vision,14/16,13/16,1.000
(all),41/80,55/80,0.036


## Links nobody asked for

The control task asks for a code on the page. Any link there is noise the sentence may have invited.

In [8]:
ctl = ok[ok.task == "control"]
ctl.groupby(["model", "sentence"]).agg(runs=("linked", "size"), linked=("linked", "sum"), passed=("passed", "sum")).unstack("sentence")

runs    linked    passed   
sentence                      off on    off on    off on
model                                                   
deepseek.deepseek-flash         4  4      0  0      4  4
gemma4:31b                      4  4      0  0      4  4
gemma4:e4b                      4  4      0  0      4  1
openrouter.openai/gpt-6-luna    4  4      0  0      4  4
qwen3.8-flash-next:vision       4  4      0  0      4  2

### The control, run again

The first sweep's control failed 5 of 20 times with the sentence and 0 of 20 without, none of them by linking. It was run
again (`--only task=control --repeats 4`, at the commit that ships the sentence; both arms still set the constant), and
the sweep moved the first runs to its `history/`. Pinned separately, so `runs.json` keeps the first sweep.

In [9]:
RERUN = HERE / "data" / "links-prompt" / "control-rerun.json"

def read_control_rerun():
    rows = []
    for cell in sorted((wmlnb.find_sweep(SWEEP) / "control").glob("*/r*/cell.json")):
        c = json.loads(cell.read_text()); m = c.get("measurement") or {}
        answer = m.get("finalAnswer") or m.get("answer") or ""
        rows.append({"model": c["combo"]["model"], "sentence": c["combo"]["sentence"], "repeat": c["repeat"],
                     "passed": bool(m.get("succeeded")), "error": bool(m.get("error")), "linked": bool(links_in(answer)),
                     "answer": answer})
    return rows

rerun = pd.DataFrame(wmlnb.extract(RERUN, read_control_rerun, source={"sweep": SWEEP, "task": "control", "rerun": True}))
both = pd.concat([ctl.assign(run="first")[["run", "model", "sentence", "passed", "linked"]],
                  rerun[~rerun.error].assign(run="again")[["run", "model", "sentence", "passed", "linked"]]])
both.groupby(["run", "sentence"]).agg(runs=("passed", "size"), failed=("passed", lambda x: int((~x).sum())), linked=("linked", "sum")).unstack("run")

runs       failed       linked      
run      again first  again first  again first
sentence                                      
off         20    20      2     0      0     0
on          20    20      2     5      0     0

In [10]:
rerun[~rerun.passed][["model", "sentence", "answer"]].assign(answer=lambda d: d.answer.str.slice(0, 80))

,model,sentence,answer
16,gemma4:e4b,off,```javascript\nconst crossDomainTest = async (...
17,gemma4:e4b,off,http://127.0.0.1:54306
21,gemma4:e4b,on,"```javascript\nconst user = ""username"";\nconst..."
23,gemma4:e4b,on,```javascript\ncrosspage-9471\n```


## How the off arm finds the form

Runs that called `agent_api_docs` at least once, and of those how many linked: without the sentence, the docs are the
only place the form is written.

In [11]:
asks.assign(read_docs=asks.docs_calls > 0).groupby(["sentence", "read_docs"]).agg(runs=("linked", "size"), linked=("linked", "sum"), right_place=("right_place", "sum"))

runs  linked  right_place
sentence read_docs                           
off      False        22       0            0
         True         58      41           41
on       False        11       0            0
         True         69      58           55

## What the sentence costs

The first call's prompt tokens are the system prompt plus the tools plus the task, so on minus off is the sentence in
each model's own tokenizer (the median over that model's runs; tasks differ only in the task text, which both arms
share). Every call of every run pays it.

In [12]:
first = ok.groupby(["model", "sentence"]).first_prompt_tokens.median().unstack("sentence")
first["sentence_tokens"] = first["on"] - first["off"]
first["calls_per_run"] = ok.groupby("model").calls.mean().round(1)
first["tokens_per_run"] = (first.sentence_tokens * first.calls_per_run).round(0)
first

sentence,off,on,sentence_tokens,calls_per_run,tokens_per_run
model,,,,,
deepseek.deepseek-flash,12557.0,12596.0,39.0,4.2,164.0
gemma4:31b,12023.0,12063.0,40.0,3.7,148.0
gemma4:e4b,12023.0,12063.0,40.0,3.6,144.0
openrouter.openai/gpt-6-luna,10253.0,10289.0,36.0,5.0,180.0
qwen3.8-flash-next:vision,12474.0,12510.0,36.0,5.0,180.0


## Unresolved addresses

Every link whose address does not parse (it renders as its label), with the model and arm that wrote it.

In [13]:
bad = [{"model": r.model, "sentence": r.sentence, "task": r.task, "address": l["address"]} for r in ok.itertuples() for l in r.links if not l["route"]]
pd.DataFrame(bad, columns=["model", "sentence", "task", "address"])

,model,sentence,task,address
0,qwen3.8-flash-next:vision,on,readonly,setting/advanced/javascript
1,qwen3.8-flash-next:vision,on,shell,setting/extension/advanced/shell
2,qwen3.8-flash-next:vision,on,shell,setting/advanced/shell
